In [1]:
from datasets import load_dataset

dataset = load_dataset("Tralalabs/simple-english-wikipedia")
docs = dataset["train"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


wiki.parquet:   0%|          | 0.00/478M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/274322 [00:00<?, ? examples/s]

In [3]:
docs = docs.shuffle(seed=42).select(range(2000))

In [4]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

In [5]:
import json

raw_docs = [doc["title"] + " " + doc["text"] for doc in docs]

with open("data/raw/raw_data.json", "w") as f:
    json.dump(raw_docs, f)

In [6]:
import re
import string
import json
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # lowercase
    text = text.lower()

    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # tokenize
    tokens = word_tokenize(text)

    # remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
nltk.download('punkt')
nltk.download('punkt_tab')

cleaned_docs = [preprocess_text(doc) for doc in raw_docs]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [8]:
with open("data/processed/cleaned_data.json", "w") as f:
    json.dump(cleaned_docs, f)

In [10]:
queries = [
    "food delivery problem",
    "climate change effects",
    "machine learning applications",
    "health benefits of exercise",
    "economic inflation causes"
]

with open("queries.txt", "w") as f:
    for q in queries:
        f.write(q + "\n")

In [15]:
import json

with open("data/processed/cleaned_data.json", "r") as f:
    documents = json.load(f)

In [16]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

class TFIDFEngine:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()

    # Fit on documents
    def fit(self, documents):
        tfidf_matrix = self.vectorizer.fit_transform(documents)
        return tfidf_matrix

    # Transform query
    def transform_query(self, query):
        return self.vectorizer.transform([query])

    # Save model
    def save(self, vectorizer_path="vectorizer.pkl"):
        with open(vectorizer_path, "wb") as f:
            pickle.dump(self.vectorizer, f)

    # Load model
    def load(self, vectorizer_path="vectorizer.pkl"):
        with open(vectorizer_path, "rb") as f:
            self.vectorizer = pickle.load(f)

In [17]:
engine = TFIDFEngine()

tfidf_matrix = engine.fit(documents)

In [18]:
engine.save("vectorizer.pkl")

In [19]:
import numpy as np
import scipy.sparse as sp

sp.save_npz("tfidf_matrix.npz", tfidf_matrix)

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query, tfidf_matrix, engine, documents, top_k=5):
    # 1. Convert query to vector
    query_vec = engine.transform_query(query)

    # 2. Compute cosine similarity
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]

    # 3. Rank documents (highest similarity first)
    ranked_indices = np.argsort(scores)[::-1]

    # 4. Get Top-K results
    top_indices = ranked_indices[:top_k]

    # 5. Build results
    results = []
    for idx in top_indices:
        results.append({
            "document": documents[idx],
            "score": float(scores[idx]),
            "title": documents[idx].split(" ")[0]
        })

    return results

In [21]:
query = "machine learning applications"

results = search(
    query=query,
    tfidf_matrix=tfidf_matrix,
    engine=engine,
    documents=documents,
    top_k=5
)

for i, r in enumerate(results):
    print(f"\nRank {i+1}")
    print("Score:", r["score"])
    print("Title:", r["title"])
    print("Text:", r["document"][:300])


Rank 1
Score: 0.1599010985035016
Title: text
Text: text speech listenfilenamejärdautropoggtitleautomatic announcementdescriptiona synthetic voice announcing arriving train swedenformatoggtext speech tts use software create sound output form spoken voice program used programs change text page audio output spoken voice normally text speech engine blin

Rank 2
Score: 0.1306480461369939
Title: apf
Text: apf imagination machine copyeditdatemarch 2026no sourcesdatemarch 2026 apf imagination machine 1979 computer apf 2 separate components apf mp1000 video game system imagination machines system motorola 6800 motorola 6847 graphics one sound channel 5 octaveoctaves produce sound 9 kilobytekb ram 14 kb 

Rank 3
Score: 0.11100833934505228
Title: heartlung
Text: heartlung transplant heartlung transplant type surgery help people sick hearts lungs surgery patients heart lungs taken replaced healthy organ anatomyorgans surgery machine called heartlung machine pumps blood patient breathes many heart

In [22]:
def precision_at_k(query, top_docs, k=5):
    relevant = 0

    for i in range(k):
        print("\n--- Document", i+1, "---")
        print(top_docs[i])

        label = int(input("Relevant? (1/0): "))
        relevant += label

    precision = relevant / k
    return precision